# AudioLDM Finetuning on new_data (Minecraft Sounds)

This notebook:
1. Prepares the `new_data/` dataset with 70/20/10 train/val/test splits
2. Visualizes CLAP embeddings **before** finetuning
3. Runs finetuning with regular checkpoints and loss logging
4. Plots training/validation losses
5. Visualizes CLAP embeddings **after** finetuning

## 0. Setup

In [41]:
import os
import sys

# Make sure we run from repo root
REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname("__file__"), "."))
os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)
print("Working directory:", os.getcwd())

Working directory: /home/dulat-rakhymkul/Documents/GitHub/AudioLDM-training-finetuning


In [42]:
import json
import random
import shutil
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import torch
import yaml

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cuda


In [43]:
# Key paths
NEW_DATA_DIR        = Path("new_data")
PREPROCESSED_DIR    = NEW_DATA_DIR / "preprocessed"   # 556 wav files
METADATA_JSON       = NEW_DATA_DIR / "metadata.json"

METADATA_OUT_DIR    = Path("data/dataset/metadata/new_minecraft")
DATASET_ROOT_JSON   = Path("data/dataset/metadata/dataset_root_new.json")

FINETUNE_CONFIG     = Path("new_data/finetune_config.yaml")
PRETRAINED_CKPT     = Path("data/checkpoints/audioldm-s-full")
CLAP_CKPT           = Path("data/checkpoints/clap_htsat_tiny.pt")

assert PREPROCESSED_DIR.exists(), f"Missing: {PREPROCESSED_DIR}"
assert METADATA_JSON.exists(),    f"Missing: {METADATA_JSON}"
assert PRETRAINED_CKPT.exists(),  f"Missing pretrained ckpt: {PRETRAINED_CKPT}"
assert CLAP_CKPT.exists(),        f"Missing CLAP ckpt: {CLAP_CKPT}"
print("All paths OK.")

All paths OK.


## 1. Data Preparation — 70 / 20 / 10 Split

In [44]:
with open(METADATA_JSON) as f:
    raw_meta = json.load(f)

print(f"Total entries in metadata.json: {len(raw_meta)}")

Total entries in metadata.json: 1457


In [45]:
# Build dataset entries
# The metadata paths point to Google Drive; we replace them with local preprocessed/ paths.
# Multiple OGG entries can map to the same WAV (e.g. MOB_ALLAY_ITEM_GIVEN1.ogg → mob_allay_item given.wav)

preprocessed_files = {f.name.lower(): f.name for f in PREPROCESSED_DIR.glob("*.wav")}

entries = []
skipped = 0

for ogg_name, meta in raw_meta.items():
    wav_fname = Path(meta["path"]).name          # e.g. "mob_allay_item given.wav"
    if wav_fname.lower() not in preprocessed_files:
        skipped += 1
        continue
    entries.append({
        "wav": wav_fname,          # relative to root (PREPROCESSED_DIR)
        "caption_0": meta["prompt"],
        "tags": meta["tags"],
    })

print(f"Usable entries: {len(entries)}  |  skipped (wav missing): {skipped}")

Usable entries: 1457  |  skipped (wav missing): 0


In [46]:
# Shuffle and split 70 / 20 / 10
random.shuffle(entries)
n = len(entries)
n_train = int(0.70 * n)
n_val   = int(0.20 * n)
n_test  = n - n_train - n_val

train_entries = entries[:n_train]
val_entries   = entries[n_train : n_train + n_val]
test_entries  = entries[n_train + n_val :]

print(f"Train: {len(train_entries)}  Val: {len(val_entries)}  Test: {len(test_entries)}")

Train: 1019  Val: 291  Test: 147


In [47]:
# Write AudioDataset-compatible JSONs  {"data": [{"wav": ..., "caption_0": ...}]}
METADATA_OUT_DIR.mkdir(parents=True, exist_ok=True)

def _strip_tags(entries):
    return [{k: v for k, v in e.items() if k != "tags"} for e in entries]

splits = {
    "train": train_entries,
    "val":   val_entries,
    "test":  test_entries,
}
for split, data in splits.items():
    out_path = METADATA_OUT_DIR / f"new_minecraft_{split}.json"
    with open(out_path, "w") as f:
        json.dump({"data": _strip_tags(data)}, f, indent=2)
    print(f"Wrote {out_path}  ({len(data)} samples)")

# Write dataset_root.json
dataset_root = {
    "new_minecraft": str(PREPROCESSED_DIR.resolve()),
    "metadata": {
        "path": {
            "new_minecraft": {
                "train": str(METADATA_OUT_DIR / "new_minecraft_train.json"),
                "val":   str(METADATA_OUT_DIR / "new_minecraft_val.json"),
                "test":  str(METADATA_OUT_DIR / "new_minecraft_test.json"),
            }
        }
    },
}
with open(DATASET_ROOT_JSON, "w") as f:
    json.dump(dataset_root, f, indent=2)
print(f"Wrote {DATASET_ROOT_JSON}")

Wrote data/dataset/metadata/new_minecraft/new_minecraft_train.json  (1019 samples)
Wrote data/dataset/metadata/new_minecraft/new_minecraft_val.json  (291 samples)
Wrote data/dataset/metadata/new_minecraft/new_minecraft_test.json  (147 samples)
Wrote data/dataset/metadata/dataset_root_new.json


## 2. Create Finetuning Config

In [48]:
with open("audioldm_train/config/2023_08_23_reproduce_audioldm/audioldm_original.yaml") as f:
    cfg = yaml.safe_load(f)

# Point to new dataset
cfg["metadata_root"] = str(DATASET_ROOT_JSON)
cfg["data"]["train"] = ["new_minecraft"]
cfg["data"]["val"]   = "new_minecraft"
cfg["data"]["test"]  = "new_minecraft"

# Dataset size: ~1020 train samples, batch 3 → 340 steps/epoch → 30 epochs = 10200 steps
BATCH_SIZE      = 3
TRAIN_SAMPLES   = 1020
EPOCHS          = 50
STEPS_PER_EPOCH = TRAIN_SAMPLES // BATCH_SIZE
MAX_STEPS       = STEPS_PER_EPOCH * EPOCHS
print(f"Steps/epoch: {STEPS_PER_EPOCH}  |  Total steps ({EPOCHS} epochs): {MAX_STEPS}")

cfg["model"]["params"]["base_learning_rate"] = 5e-5
cfg["model"]["params"]["warmup_steps"]        = 200
cfg["model"]["params"]["batchsize"]            = BATCH_SIZE
cfg["step"]["max_steps"]                       = MAX_STEPS
cfg["step"]["validation_every_n_epochs"]       = 5
cfg["step"]["save_checkpoint_every_n_steps"]   = 500
# save_top_k controls the best-val-loss ModelCheckpoint callback in the training script.
# Set to 1 so only the single best val/loss_simple checkpoint is kept.
# Periodic step checkpoints (every 500 steps) are saved separately regardless.
cfg["step"]["save_top_k"]                      = 1

cfg["log_directory"] = "./log/latent_diffusion"
cfg["project"]       = "audioldm_finetune_new_data"

with open(FINETUNE_CONFIG, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

print(f"Config written to {FINETUNE_CONFIG}")
print(f"Checkpoints: periodic every 500 steps  +  best val/loss_simple (top-1)")

Steps/epoch: 340  |  Total steps (50 epochs): 17000
Config written to new_data/finetune_config.yaml
Checkpoints: periodic every 500 steps  +  best val/loss_simple (top-1)


## 3. Embedding Visualization — **Before** Finetuning

We extract CLAP text embeddings for all prompts in the dataset and visualize them with t-SNE,
colouring by top-level tag (e.g. `mob`, `ambient`, `block`, …).

In [49]:
from audioldm_train.conditional_models import CLAPAudioEmbeddingClassifierFreev2

clap_model = CLAPAudioEmbeddingClassifierFreev2(
    pretrained_path=str(CLAP_CKPT),
    sampling_rate=16000,
    embed_mode="text",
    amodel="HTSAT-tiny",
)
clap_model.eval().to(DEVICE)
print("CLAP model loaded.")

/home/dulat-rakhymkul/Documents/GitHub/AudioLDM-training-finetuning/myenv/lib/python3.10/site-packages/torchlibrosa/stft.py:193: FutureWarning: Pass size=1024 as keyword args. From version 0.10 passing these as positional arguments will result in an error
  fft_window = librosa.util.pad_center(fft_window, n_fft)
Some weights of the model checkpoint at roberta-base were not used when initializing RobertaModel: ['lm_head.layer_norm.bias', 'lm_head.bias', 'lm_head.dense.bias', 'lm_head.dense.weight', 'lm_head.layer_norm.weight']
- This IS expected if you are initializing RobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weig

CLAP model loaded.


/home/dulat-rakhymkul/Documents/GitHub/AudioLDM-training-finetuning/myenv/lib/python3.10/site-packages/torchaudio/transforms/_transforms.py:580: UserWarning: Argument 'onesided' has been deprecated and has no influence on the behavior of this module.
  warnings.warn(


In [50]:
@torch.no_grad()
def extract_text_embeddings(model, texts, batch_size=64):
    """Return (N, 512) numpy array of CLAP text embeddings."""
    all_embeds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        # CLAPAudioEmbeddingClassifierFreev2.forward expects a list of strings
        # It returns [B, 1, 512] — we squeeze the middle dim
        emb = model(batch)           # [B, 1, 512]
        all_embeds.append(emb.squeeze(1).cpu().numpy())
    return np.concatenate(all_embeds, axis=0)

In [51]:
# Collect prompts + labels from all splits
all_prompts = [e["caption_0"] for e in entries]
all_tags    = [e["tags"][0] if e["tags"] else "unknown" for e in entries]

print(f"Extracting embeddings for {len(all_prompts)} prompts …")
embeddings_before = extract_text_embeddings(clap_model, all_prompts)
print(f"Embedding matrix shape: {embeddings_before.shape}")

Extracting embeddings for 1457 prompts …
Embedding matrix shape: (1457, 512)


In [52]:
from sklearn.manifold import TSNE
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
tag_ids = le.fit_transform(all_tags)
n_classes = len(le.classes_)
print(f"Unique top-level tags: {le.classes_}")

print("Running t-SNE …")
tsne = TSNE(n_components=2, perplexity=40, random_state=42, n_iter=1000)
coords_before = tsne.fit_transform(embeddings_before)
print("t-SNE done.")

Unique top-level tags: ['ambient' 'block' 'enchant' 'entity' 'event' 'fire' 'fireworks' 'item'
 'liquid' 'minecart' 'mob' 'music' 'note' 'portal' 'random' 'records' 'ui']
Running t-SNE …
t-SNE done.


In [53]:
def plot_embeddings(coords, tag_ids, classes, title, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 8))
    cmap = plt.get_cmap("tab20", len(classes))
    sc = ax.scatter(coords[:, 0], coords[:, 1],
                    c=tag_ids, cmap=cmap, s=10, alpha=0.7)
    patches = [mpatches.Patch(color=cmap(i), label=cls) for i, cls in enumerate(classes)]
    ax.legend(handles=patches, bbox_to_anchor=(1.01, 1), loc="upper left",
              fontsize=8, ncol=1)
    ax.set_title(title, fontsize=14)
    ax.set_xlabel("t-SNE dim 1")
    ax.set_ylabel("t-SNE dim 2")
    ax.set_xticks([]); ax.set_yticks([])
    return ax

fig, ax = plt.subplots(figsize=(12, 9))
plot_embeddings(coords_before, tag_ids, le.classes_,
                "CLAP Text Embeddings — Before Finetuning", ax=ax)
plt.tight_layout()
plt.savefig("new_data/embeddings_before.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved new_data/embeddings_before.png")

Saved new_data/embeddings_before.png


## 4. Finetuning

Launches the standard training script with our new config, loading the pretrained checkpoint.

In [54]:
# Verify the command before running
LOG_FILE = Path("new_data/train.log")

cmd = (
    f"python3 -u audioldm_train/train/latent_diffusion.py "
    f"-c {FINETUNE_CONFIG} "
    f"--reload_from_ckpt {PRETRAINED_CKPT} "
    f"2>&1 | tee {LOG_FILE}"
)
print("Command to run:\n", cmd)
print(f"\nAll output will stream below AND be saved to {LOG_FILE}")

Command to run:
 python3 -u audioldm_train/train/latent_diffusion.py -c new_data/finetune_config.yaml --reload_from_ckpt data/checkpoints/audioldm-s-full 2>&1 | tee new_data/train.log

All output will stream below AND be saved to new_data/train.log


In [55]:
import subprocess, sys, os

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"   # prevents Python from buffering stdout in the child process

process = subprocess.Popen(
    cmd,
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

for line in process.stdout:
    sys.stdout.write(line)
    sys.stdout.flush()

process.wait()
print(f"\nTraining finished with return code {process.returncode}")

SEED EVERYTHING TO 0
Seed set to 0
Add-ons: []
Build dataset split train from ['new_minecraft']
Data size: 1019
/home/dulat-rakhymkul/Documents/GitHub/AudioLDM-training-finetuning/audioldm_train/utilities/audio/stft.py:42: FutureWarning: Pass size=1024 as keyword args. From version 0.10 passing these as positional arguments will result in an error
  fft_window = pad_center(fft_window, filter_length)
/home/dulat-rakhymkul/Documents/GitHub/AudioLDM-training-finetuning/audioldm_train/utilities/audio/stft.py:145: FutureWarning: Pass sr=16000, n_fft=1024, n_mels=64, fmin=0, fmax=8000 as keyword args. From version 0.10 passing these as positional arguments will result in an error
  mel_basis = librosa_mel_fn(
Dataset initialize finished
The length of the dataset is 1019, the length of the dataloader is 340, the batchsize is 3
Add-ons: []
Build dataset split test from new_minecraft
Data size: 147
Dataset initialize finished
Copying test subset data to ./log/testset_data/new_minecraft

100%|██

## 5. Loss Curves

In [68]:
import glob

RUN_DIR     = Path("log/latent_diffusion/new_data/finetune_config")
CKPT_DIR    = RUN_DIR / "checkpoints"
CSV_LOG_DIR = RUN_DIR / "csv_logs"

print(f"Run directory  : {RUN_DIR}")
print(f"Checkpoints    : {CKPT_DIR}")
print(f"CSV logs       : {CSV_LOG_DIR}")

csv_files = sorted(CSV_LOG_DIR.glob("**/metrics.csv"))
if not csv_files:
    print("\nNo CSV logs found yet — run the training cell first.")
    df = None
else:
    print(f"\nFound {len(csv_files)} version(s): {[str(f) for f in csv_files]}")
    frames = [pd.read_csv(f) for f in csv_files]
    df = pd.concat(frames, ignore_index=True)
    df = df.sort_values("step").reset_index(drop=True)
    print(f"Total rows across all versions: {len(df)}")
    df.head()

Run directory  : log/latent_diffusion/new_data/finetune_config
Checkpoints    : log/latent_diffusion/new_data/finetune_config/checkpoints
CSV logs       : log/latent_diffusion/new_data/finetune_config/csv_logs

Found 2 version(s): ['log/latent_diffusion/new_data/finetune_config/csv_logs/version_0/metrics.csv', 'log/latent_diffusion/new_data/finetune_config/csv_logs/version_1/metrics.csv']
Total rows across all versions: 403


In [69]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Training loss (epoch-level) ---
train_col = "train/loss_simple_epoch"
if train_col in df.columns:
    train_df = df[["epoch", train_col]].dropna().drop_duplicates("epoch")
    axes[0].plot(train_df["epoch"], train_df[train_col], marker="o", linewidth=2, label=train_col)
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("Training Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

# --- Validation loss (epoch-level) ---
val_col = "val/loss_simple"
if val_col in df.columns:
    val_df = df[["epoch", val_col]].dropna().drop_duplicates("epoch")
    axes[1].plot(val_df["epoch"], val_df[val_col], marker="o", linewidth=2, label=val_col)
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].set_title("Validation Loss")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

plt.suptitle("Finetuning Loss — new_data (70/20/10 split)", fontsize=14)
plt.tight_layout()
plt.savefig("new_data/finetuning_loss.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved new_data/finetuning_loss.png")

Saved new_data/finetuning_loss.png


In [70]:
# Show all available loss columns and their final values
loss_cols = [c for c in df.columns if "loss" in c.lower()]
summary = {}
for col in loss_cols:
    last_val = df[col].dropna().iloc[-1] if not df[col].dropna().empty else None
    summary[col] = last_val
pd.Series(summary, name="final_value").to_frame()

,final_value
train/loss_simple_step,0.009709
train/loss_vlb_step,0.000049
train/loss_step,0.009709
train/loss_vlb_epoch,0.002069
train/loss_simple_epoch,0.133037
train/loss_epoch,0.133037
val/loss_vlb,0.001337
val/loss_simple,0.113060
val/loss,0.113060


## 6. Embedding Visualization — **After** Finetuning

Load the best finetuned checkpoint and re-extract CLAP embeddings from the model's conditioning
encoder to compare how the representation space shifted.

In [65]:
from audioldm_train.utilities.model_util import instantiate_from_config

# Best val loss checkpoint is named best-valloss=*
best_ckpts = sorted(CKPT_DIR.glob("best-valloss=*.ckpt"))

if not best_ckpts:
    print(f"No best-val-loss checkpoint found in {CKPT_DIR}. Run training first.")
    FINETUNE_CKPT = None
else:
    # Pick the one with the lowest val loss (encoded in filename)
    FINETUNE_CKPT = str(best_ckpts[0])
    print(f"Best val-loss checkpoint: {FINETUNE_CKPT}")

Best val-loss checkpoint: log/latent_diffusion/new_data/finetune_config/checkpoints/best-valloss=0.1153-step=10199.ckpt


In [60]:
if FINETUNE_CKPT is None:
    print("Skipping — no checkpoint available.")
else:
    with open(FINETUNE_CONFIG) as f:
        ft_cfg = yaml.safe_load(f)

    ft_model = instantiate_from_config(ft_cfg["model"])

    ckpt_data = torch.load(FINETUNE_CKPT, map_location="cpu")
    state_dict = ckpt_data["state_dict"]
    # Filter keys whose shapes mismatch (safety net)
    model_sd   = ft_model.state_dict()
    filtered   = {k: v for k, v in state_dict.items()
                  if k in model_sd and v.shape == model_sd[k].shape}
    missing    = ft_model.load_state_dict(filtered, strict=False)
    print(f"Loaded {len(filtered)}/{len(model_sd)} keys.  Missing: {len(missing.missing_keys)}")

    ft_model.eval().to(DEVICE)

    # Extract the CLAP cond encoder from the finetuned model
    clap_idx = ft_model.cond_stage_model_metadata["film_clap_cond1"]["model_idx"]
    clap_ft = ft_model.cond_stage_models[clap_idx]
    clap_ft.eval().to(DEVICE)
    print("Finetuned CLAP encoder ready.")

LatentDiffusion: Running in eps-prediction mode


/home/dulat-rakhymkul/Documents/GitHub/AudioLDM-training-finetuning/myenv/lib/python3.10/site-packages/torchlibrosa/stft.py:193: FutureWarning: Pass size=1024 as keyword args. From version 0.10 passing these as positional arguments will result in an error
  fft_window = librosa.util.pad_center(fft_window, n_fft)
Some weights of the model checkpoint at roberta-base were not used when initializing RobertaModel: ['lm_head.layer_norm.bias', 'lm_head.bias', 'lm_head.dense.bias', 'lm_head.dense.weight', 'lm_head.layer_norm.weight']
- This IS expected if you are initializing RobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weig

+ Use extra condition on UNet channel using Film. Extra condition dimension is 512. 
DiffusionWrapper has 185.04 M params.
Keeping EMAs of 692.
making attention of type 'vanilla' with 512 in_channels
making attention of type 'vanilla' with 512 in_channels


/home/dulat-rakhymkul/Documents/GitHub/AudioLDM-training-finetuning/myenv/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/dulat-rakhymkul/Documents/GitHub/AudioLDM-training-finetuning/myenv/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/home/dulat-rakhymkul/Documents/GitHub/AudioLDM-training-finetuning/myenv/lib/python3.10/site-packages/taming/modules/losses/lpips.py:29: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default

loaded pretrained LPIPS loss from taming/modules/autoencoder/lpips/vgg.pth


/home/dulat-rakhymkul/Documents/GitHub/AudioLDM-training-finetuning/audioldm_train/utilities/model_util.py:282: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.lo

Removing weight norm...
Initial learning rate 1e-05
--> Reload weight of autoencoder from data/checkpoints/vae_mel_16k_64bins.ckpt


Some weights of the model checkpoint at roberta-base were not used when initializing RobertaModel: ['lm_head.layer_norm.bias', 'lm_head.bias', 'lm_head.dense.bias', 'lm_head.dense.weight', 'lm_head.layer_norm.weight']
- This IS expected if you are initializing RobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.weight', 'roberta.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_472423/1533382833

Loaded 2777/2777 keys.  Missing: 0
Finetuned CLAP encoder ready.


In [61]:
if FINETUNE_CKPT is not None:
    print(f"Extracting post-finetuning embeddings for {len(all_prompts)} prompts …")
    embeddings_after = extract_text_embeddings(clap_ft, all_prompts)
    print(f"Embedding matrix shape: {embeddings_after.shape}")

    print("Running t-SNE …")
    coords_after = TSNE(n_components=2, perplexity=40, random_state=42, n_iter=1000).fit_transform(embeddings_after)
    print("t-SNE done.")

Extracting post-finetuning embeddings for 1457 prompts …
Embedding matrix shape: (1457, 512)
Running t-SNE …
t-SNE done.


In [62]:
if FINETUNE_CKPT is not None:
    fig, axes = plt.subplots(1, 2, figsize=(22, 9))

    plot_embeddings(coords_before, tag_ids, le.classes_,
                    "CLAP Text Embeddings — Before Finetuning", ax=axes[0])
    plot_embeddings(coords_after,  tag_ids, le.classes_,
                    "CLAP Text Embeddings — After Finetuning",  ax=axes[1])

    plt.suptitle("Embedding Space Comparison", fontsize=16)
    plt.tight_layout()
    plt.savefig("new_data/embeddings_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved new_data/embeddings_comparison.png")

Saved new_data/embeddings_comparison.png


### Cosine Similarity Analysis

In [63]:
if FINETUNE_CKPT is not None:
    from sklearn.metrics.pairwise import cosine_similarity

    # For each class, compute mean intra-class cosine similarity before and after
    results = []
    for cls_idx, cls_name in enumerate(le.classes_):
        mask = (tag_ids == cls_idx)
        if mask.sum() < 2:
            continue
        emb_b = embeddings_before[mask]
        emb_a = embeddings_after[mask]
        sim_b = cosine_similarity(emb_b).mean()
        sim_a = cosine_similarity(emb_a).mean()
        results.append({"class": cls_name, "n": mask.sum(),
                        "intra_cos_before": sim_b, "intra_cos_after": sim_a,
                        "delta": sim_a - sim_b})

    sim_df = pd.DataFrame(results).sort_values("delta", ascending=False)
    display(sim_df.round(4))

    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(sim_df))
    ax.bar(x - 0.2, sim_df["intra_cos_before"], width=0.4, label="Before", alpha=0.8)
    ax.bar(x + 0.2, sim_df["intra_cos_after"],  width=0.4, label="After",  alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(sim_df["class"], rotation=45, ha="right")
    ax.set_ylabel("Mean Intra-class Cosine Similarity")
    ax.set_title("Embedding Cohesion per Class — Before vs After Finetuning")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig("new_data/cosine_similarity_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()

,class,n,intra_cos_before,intra_cos_after,delta
4,event,7,0.6022,0.7426,0.1404
13,records,19,0.5433,0.6619,0.1186
2,enchant,13,0.7555,0.8666,0.1111
9,mob,662,0.5371,0.5662,0.0291
1,block,333,0.5084,0.5313,0.0228
0,ambient,188,0.4831,0.4970,0.0139
5,fireworks,7,0.6326,0.6352,0.0027
8,minecart,5,0.7730,0.7730,0.0000
11,portal,3,0.8663,0.8663,0.0000
14,ui,3,0.8487,0.8487,0.0000


## 7. Quick Inference Check (Optional)

Generate a few samples from the finetuned model to do a sanity-check.

In [71]:
if FINETUNE_CKPT is not None:
    test_prompts = [
        "generate me minecraft sound where mob ghast scream",
        "generate me minecraft sound where mob villager say",
    ]

    infer_cmd = (
        f"python3 audioldm_train/infer.py "
        f"--config_yaml {FINETUNE_CONFIG} "
        f"--list_inference new_data/test_prompts.txt "
        f"--reload_from_ckpt {FINETUNE_CKPT}"
    )

    with open("new_data/test_prompts.txt", "w") as f:
        f.write("\n".join(test_prompts))

    print("Inference command:\n", infer_cmd)
    print("\nUncomment the subprocess.run line to actually run inference.")
    # result = subprocess.run(infer_cmd, shell=True, stdout=sys.stdout, stderr=subprocess.STDOUT)
    # print(f"Inference finished with return code {result.returncode}")

Inference command:
 python3 audioldm_train/infer.py --config_yaml new_data/finetune_config.yaml --list_inference new_data/test_prompts.txt --reload_from_ckpt log/latent_diffusion/new_data/finetune_config/checkpoints/best-valloss=0.1153-step=10199.ckpt

Uncomment the subprocess.run line to actually run inference.


## Summary

| Step | Output |
|---|---|
| Data split | `data/dataset/metadata/new_minecraft/{train,val,test}.json` |
| Config | `new_data/finetune_config.yaml` |
| Checkpoints | `log/latent_diffusion/audioldm_finetune_new_data/.../checkpoints/` |
| Loss plots | `new_data/finetuning_loss.png` |
| Embedding (before) | `new_data/embeddings_before.png` |
| Embedding (comparison) | `new_data/embeddings_comparison.png` |
| Cosine similarity | `new_data/cosine_similarity_comparison.png` |